# Multi-Agent Collaboration

The **Multi-Agent Collaboration** pattern structures a system as a cooperative ensemble of distinct, specialized agents. Instead of one monolithic agent trying to do everything, a complex goal is decomposed into sub-problems — each assigned to an agent with the right tools, knowledge, and role for that specific task.

The result is more than just division of labor: the collective performance of a well-designed multi-agent system surpasses what any single agent could achieve alone.

**Collaboration models:**
- **Sequential handoff** — Agent A completes its task and passes output to Agent B (pipeline)
- **Parallel processing** — Multiple agents work simultaneously; results are combined
- **Supervisor / hierarchical** — A coordinator agent delegates to specialized worker agents
- **Critic-reviewer** — One agent creates output; another evaluates it for quality or compliance

**Use cases:** research + writing pipelines, software development (requirements → code → test → docs), financial analysis (data fetch → sentiment → technical → recommendation), customer support escalation, competitive intelligence.

## Implementation with Flyte v2

This notebook implements **two collaboration models** using Flyte v2 primitives:

1. **Sequential handoff** — Researcher agent → Writer agent (blog post pipeline)
2. **Parallel specialist review** — Draft → [Fact-checker ‖ SEO optimizer] → Synthesizer

Each agent is a Flyte **task**. Agent interactions are explicit data edges in a Flyte **workflow** — making the collaboration structure visible in the DAG view, retryable per-agent, and fully auditable.

#### CrewAI vs Flyte v2 — Key Differences

| Aspect | CrewAI | Flyte v2 |
|--------|--------|----------|
| **Agent definition** | `Agent(role=..., goal=..., backstory=...)` | Async Flyte task with typed input/output |
| **Data passing** | Implicit (agent context / string passing) | Typed dataclasses — fully serialized between tasks |
| **Collaboration structure** | `Process.sequential` / `Process.hierarchical` | Explicit workflow DAG — visible in UI |
| **Parallelism** | Sequential by default | `asyncio.gather` in a task, or parallel Flyte tasks in the workflow |
| **Retry granularity** | Entire crew re-runs | Per-task retries — only the failed agent re-runs |
| **Observability** | `verbose=True` logs | Structured typed outputs per agent + live report tab |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic

### 2. Export your API key

In [ ]:
%env ANTHROPIC_API_KEY=sk-ant-...

### 3. Import dependencies and configure the Flyte TaskEnvironment

In [ ]:
from __future__ import annotations

import asyncio
import os
from dataclasses import dataclass, field
from datetime import timedelta

import anthropic
import flyte
import flyte.report

flyte.init(
    endpoint="<your-union-endpoint-url>",
    org="<your-union-org>",
    project="<your-project>",
    domain="development",
    image_builder="remote",
    auth_type="DeviceFlow",
)

_image = (
    flyte.Image.from_debian_base(name="multi-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0")
)

# Shared environment for all agents in this notebook.
# ReusePolicy keeps containers warm — since agents hand off to each other
# sequentially, the next agent in the pipeline reuses the warm pod.
agent_env = flyte.TaskEnvironment(
    name="multi_agent_env",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
    reusable=flyte.ReusePolicy(
        replicas=(1, 6),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=10),
    ),
)

### 4. Define the data models

Each agent's output is a **typed dataclass** — not a raw string. This is the critical difference from string-passing frameworks:

- The downstream agent receives structured data it can reference by field name
- Flyte stores each agent's output in object storage, making the handoff durable and inspectable
- The Flyte UI shows the full typed output of each agent as a structured record
- If any agent fails and retries, it re-receives the same typed upstream output — exactly as before

In [ ]:
# ── Sequential pipeline: Researcher → Writer ───────────────────────────────────

@dataclass
class ResearchOutput:
    """
    Structured output from the Researcher agent.

    Fields are explicit so the Writer agent can reference them by name,
    not by parsing a string. This is the typed handoff that makes the
    collaboration auditable and the pipeline resilient to agent drift.
    """
    topic: str
    summary: str                        # High-level overview
    key_findings: list[str]             # Bullet-point findings
    notable_examples: list[str]         # Concrete examples to use
    recommended_angle: str              # Editorial recommendation for the writer
    sources_consulted: int              # How many knowledge areas were covered


@dataclass
class ArticleOutput:
    """
    Structured output from the Writer agent.

    Carrying metadata (word_count, target_audience) alongside the content
    allows downstream agents (editors, SEO optimizers) to make
    informed decisions without re-parsing the article body.
    """
    topic: str
    title: str
    body: str                           # The full article text
    word_count: int
    target_audience: str
    key_points: list[str]               # Main points covered (for downstream agents)


# ── Parallel review: Fact-checker ‖ SEO optimizer → Synthesizer ───────────────

@dataclass
class FactCheckResult:
    """Output from the Fact-checker agent."""
    issues_found: list[str]             # Factual problems or unsupported claims
    verified_claims: list[str]          # Claims confirmed as accurate
    overall_accuracy_score: int         # 0–100
    recommendation: str                 # "publish" | "revise" | "reject"


@dataclass
class SEOResult:
    """Output from the SEO optimizer agent."""
    suggested_title: str
    meta_description: str               # 150–160 chars
    primary_keywords: list[str]
    secondary_keywords: list[str]
    readability_score: int              # 0–100 (Flesch-Kincaid equivalent)
    suggested_improvements: list[str]   # Actionable SEO suggestions


@dataclass
class FinalArticle:
    """Final output of the complete multi-agent pipeline."""
    topic: str
    title: str
    body: str
    meta_description: str
    primary_keywords: list[str]
    fact_check_score: int
    readability_score: int
    editorial_notes: list[str]

## Part 1: Sequential Handoff Pattern

The most common multi-agent structure: Agent A completes its specialized task and passes its typed output to Agent B. The Flyte workflow makes the handoff explicit — you can see in the DAG view exactly what data flows from Researcher to Writer.

```
researcher_agent(topic) ──► ResearchOutput ──► writer_agent(topic, research) ──► ArticleOutput
```

### 5a. Researcher agent

The researcher's role is information synthesis, not writing. Its output is structured so the writer can use `research.key_findings` and `research.recommended_angle` directly — no parsing required.

In [ ]:
RESEARCHER_SYSTEM = """\
You are a Senior Research Analyst specializing in technology and business trends.
Your role is information synthesis, not writing. Produce structured research findings
that a content writer can directly use.

Be specific: include concrete examples, data points, and named entities.
Your recommended_angle should suggest a unique editorial perspective, not a generic one.
Return your analysis in this exact JSON format:
{
  "summary": "<2-3 sentence overview>",
  "key_findings": ["finding 1", "finding 2", ...],
  "notable_examples": ["example 1", "example 2", ...],
  "recommended_angle": "<specific editorial angle for maximum impact>",
  "sources_consulted": <integer>
}"""


@flyte.trace
async def _research(topic: str) -> dict:
    """Traced research LLM call."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        system=RESEARCHER_SYSTEM,
        messages=[{"role": "user", "content": f"Research topic: {topic}"}],
    )
    import json
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


@agent_env.task(
    retries=3,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
)
async def researcher_agent(topic: str) -> ResearchOutput:
    """
    Researcher agent — Phase 1 of the sequential pipeline.

    Specialized role: information synthesis and editorial strategy.
    The typed ResearchOutput ensures the Writer agent receives structured
    data, not a blob of text to parse.

    Flyte stores this output in object storage before passing it to the
    Writer. If the Writer task fails, it retries with the same ResearchOutput
    — the Researcher is not re-run.
    """
    data = await _research(topic=topic)
    return ResearchOutput(
        topic=topic,
        summary=data["summary"],
        key_findings=data["key_findings"],
        notable_examples=data["notable_examples"],
        recommended_angle=data["recommended_angle"],
        sources_consulted=data["sources_consulted"],
    )

### 5b. Writer agent

The writer receives the researcher's structured output and uses each field to produce a targeted article. Note that the writer's prompt is constructed from typed fields (`research.key_findings`, `research.recommended_angle`) — not from parsing a string. This is the key advantage of typed handoffs.

In [ ]:
WRITER_SYSTEM = """\
You are a Technical Content Writer who creates engaging, accurate articles for
a technical but non-specialist audience.
Use the research provided to write a compelling piece that follows the recommended angle.
Return your output in this exact JSON format:
{
  "title": "<compelling headline>",
  "body": "<full article text, ~500 words>",
  "word_count": <integer>,
  "target_audience": "<description of intended reader>",
  "key_points": ["main point 1", "main point 2", ...]
}"""


@flyte.trace
async def _write(topic: str, research: ResearchOutput) -> dict:
    """Traced writing LLM call. Receives typed research, not a raw string."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    # Build a structured prompt from typed research fields
    research_brief = (
        f"Topic: {topic}\n\n"
        f"Research Summary: {research.summary}\n\n"
        f"Key Findings:\n" + "\n".join(f"- {f}" for f in research.key_findings) + "\n\n"
        f"Notable Examples:\n" + "\n".join(f"- {e}" for e in research.notable_examples) + "\n\n"
        f"Recommended editorial angle: {research.recommended_angle}"
    )

    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=3000,
        system=WRITER_SYSTEM,
        messages=[{"role": "user", "content": research_brief}],
    )
    import json
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


@agent_env.task(
    retries=3,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
)
async def writer_agent(topic: str, research: ResearchOutput) -> ArticleOutput:
    """
    Writer agent — Phase 2 of the sequential pipeline.

    Specialized role: transforming structured research into engaging content.
    Receives typed ResearchOutput (not a string), so it can use research
    fields directly without parsing.
    """
    data = await _write(topic=topic, research=research)
    body = data["body"]
    return ArticleOutput(
        topic=topic,
        title=data["title"],
        body=body,
        word_count=data.get("word_count", len(body.split())),
        target_audience=data["target_audience"],
        key_points=data["key_points"],
    )

### 5c. Sequential collaboration workflow

The Flyte workflow declares the dependency explicitly. In the Flyte UI, this renders as a two-node DAG: `researcher_agent → writer_agent`. Each node shows its typed inputs and outputs.

In [ ]:
@flyte.workflow
def sequential_collaboration(topic: str) -> ArticleOutput:
    """
    Sequential handoff: Researcher → Writer.

    DAG:
      researcher_agent(topic)
        └──► ResearchOutput
               └──► writer_agent(topic, research)
                      └──► ArticleOutput

    Key properties:
    - The Writer task only starts after the Researcher completes
    - If the Writer fails, only the Writer retries (Researcher output is cached)
    - The ResearchOutput is stored and visible in the Flyte UI between tasks
    """
    research = researcher_agent(topic=topic)
    return writer_agent(topic=topic, research=research)

### 6. Run the sequential pipeline locally

In [ ]:
run = flyte.with_runcontext(mode="local").run(
    sequential_collaboration,
    topic="The rise of agentic AI systems and their impact on software development workflows",
)
run.wait()
article = run.outputs()[0]

print(f"Title: {article.title}")
print(f"Word count: {article.word_count}")
print(f"Target audience: {article.target_audience}")
print(f"\nKey points covered:")
for kp in article.key_points:
    print(f"  - {kp}")
print("\n" + "=" * 60)
print(article.body)

---

## Part 2: Parallel Specialist Review Pattern

Once a draft article exists, multiple specialists can review it **simultaneously** — fact-checker and SEO optimizer run in parallel, then a synthesizer merges their feedback into a final polished article.

```
                 ┌──► fact_checker_agent ──► FactCheckResult  ─┐
writer output ───┤                                              ├──► synthesizer_agent ──► FinalArticle
                 └──► seo_optimizer_agent ──► SEOResult ────────┘
```

In Flyte, this is expressed naturally in a workflow: two tasks with the same input and no dependency on each other execute in parallel. Flyte's scheduler dispatches them concurrently.

### 7a. Fact-checker and SEO optimizer agents

In [ ]:
import json

FACT_CHECKER_SYSTEM = """\
You are a rigorous fact-checker. Your role is to evaluate an article for factual accuracy.
Focus on: unsupported claims, incorrect statistics, misleading statements, and outdated information.
Return your analysis in this exact JSON format:
{
  "issues_found": ["issue 1", ...],
  "verified_claims": ["accurate claim 1", ...],
  "overall_accuracy_score": <0-100>,
  "recommendation": "publish" | "revise" | "reject"
}"""

SEO_SYSTEM = """\
You are an SEO specialist. Analyze an article and provide optimization recommendations.
Focus on: keyword opportunities, title optimization, meta description, and readability.
Return your analysis in this exact JSON format:
{
  "suggested_title": "<SEO-optimized title>",
  "meta_description": "<150-160 char meta description>",
  "primary_keywords": ["keyword 1", ...],
  "secondary_keywords": ["keyword 1", ...],
  "readability_score": <0-100>,
  "suggested_improvements": ["improvement 1", ...]
}"""


async def _call_specialist(system: str, content: str) -> dict:
    """Generic specialist LLM call returning parsed JSON."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        system=system,
        messages=[{"role": "user", "content": content}],
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


@agent_env.task(
    retries=3,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
)
async def fact_checker_agent(article: ArticleOutput) -> FactCheckResult:
    """
    Fact-checker agent — runs in parallel with SEO optimizer.

    Specialized role: evaluating factual accuracy and identifying unsupported claims.
    Operates independently of the SEO agent — no shared state, no coordination needed.
    Flyte schedules both parallel agents concurrently on separate pods.
    """
    content = f"Title: {article.title}\n\nArticle:\n{article.body}"
    data = await _call_specialist(FACT_CHECKER_SYSTEM, content)
    return FactCheckResult(
        issues_found=data.get("issues_found", []),
        verified_claims=data.get("verified_claims", []),
        overall_accuracy_score=data.get("overall_accuracy_score", 80),
        recommendation=data.get("recommendation", "revise"),
    )


@agent_env.task(
    retries=3,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
)
async def seo_optimizer_agent(article: ArticleOutput) -> SEOResult:
    """
    SEO optimizer agent — runs in parallel with fact-checker.

    Specialized role: keyword strategy, title optimization, readability scoring.
    Operates independently, receiving the same ArticleOutput as the fact-checker.
    """
    content = f"Title: {article.title}\n\nArticle:\n{article.body}\n\nKey points: {', '.join(article.key_points)}"
    data = await _call_specialist(SEO_SYSTEM, content)
    return SEOResult(
        suggested_title=data.get("suggested_title", article.title),
        meta_description=data.get("meta_description", ""),
        primary_keywords=data.get("primary_keywords", []),
        secondary_keywords=data.get("secondary_keywords", []),
        readability_score=data.get("readability_score", 75),
        suggested_improvements=data.get("suggested_improvements", []),
    )

### 7b. Synthesizer agent

The synthesizer receives both specialist outputs and produces the final polished article. It only starts after **both** parallel agents complete — Flyte handles this dependency automatically when the workflow declares it.

In [ ]:
SYNTHESIZER_SYSTEM = """\
You are an editorial director making final publication decisions.
You receive an article draft along with specialist reviews from a fact-checker and an SEO optimizer.
Your job: apply the recommended improvements and produce the final, publication-ready article.
Apply factual corrections, SEO improvements, and use the optimized title and meta description.
Return only the final article body text — no JSON, no preamble."""


@flyte.trace
async def _synthesize(
    article: ArticleOutput,
    fact_check: FactCheckResult,
    seo: SEOResult,
) -> str:
    """Traced synthesis call — merges parallel specialist feedback."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    editorial_brief = (
        f"Original article:\nTitle: {article.title}\n{article.body}\n\n"
        f"FACT-CHECK REPORT (score: {fact_check.overall_accuracy_score}/100, "
        f"recommendation: {fact_check.recommendation}):\n"
        + ("Issues to fix:\n" + "\n".join(f"- {i}" for i in fact_check.issues_found) if fact_check.issues_found else "No factual issues found.")
        + f"\n\nSEO REPORT (readability: {seo.readability_score}/100):\n"
        f"Optimized title: {seo.suggested_title}\n"
        f"Meta description: {seo.meta_description}\n"
        f"Primary keywords to incorporate: {', '.join(seo.primary_keywords)}\n"
        "Suggested improvements:\n"
        + "\n".join(f"- {s}" for s in seo.suggested_improvements)
    )

    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=3000,
        system=SYNTHESIZER_SYSTEM,
        messages=[{"role": "user", "content": editorial_brief}],
    )
    return response.content[0].text


@agent_env.task(
    retries=3,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def synthesizer_agent(
    article: ArticleOutput,
    fact_check: FactCheckResult,
    seo: SEOResult,
) -> FinalArticle:
    """
    Synthesizer agent — runs after both parallel specialists complete.

    Receives typed outputs from both the fact-checker and SEO optimizer
    (not concatenated strings) and merges them into the final article.
    Flyte guarantees both upstream tasks have completed before this task starts.
    """
    final_body = await _synthesize(article=article, fact_check=fact_check, seo=seo)

    editorial_notes = []
    if fact_check.issues_found:
        editorial_notes.append(f"Fact-check: {len(fact_check.issues_found)} issues corrected")
    editorial_notes.extend(seo.suggested_improvements[:3])

    # Emit a summary report visible in the Flyte UI
    def _esc(t: str) -> str:
        return t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    await flyte.report.replace.aio(
        "<html><body style='font-family:sans-serif;max-width:900px;margin:auto;padding:1.5em'>"
        "<h1>Multi-Agent Pipeline — Final Article Report</h1>"
        f"<h2>Fact-Check Score: {fact_check.overall_accuracy_score}/100 ({fact_check.recommendation})</h2>"
        + ("<ul>" + "".join(f"<li>{_esc(i)}</li>" for i in fact_check.issues_found) + "</ul>" if fact_check.issues_found else "<p>No factual issues.</p>")
        + f"<h2>SEO Score: {seo.readability_score}/100</h2>"
        f"<p>Optimized title: <strong>{_esc(seo.suggested_title)}</strong></p>"
        f"<p>Keywords: {_esc(', '.join(seo.primary_keywords))}</p>"
        "<h2>Final Article</h2>"
        f"<pre style='background:#f4f4f4;padding:1em;border-radius:4px;white-space:pre-wrap'>{_esc(final_body)}</pre>"
        "</body></html>"
    )
    await flyte.report.flush.aio()

    return FinalArticle(
        topic=article.topic,
        title=seo.suggested_title,
        body=final_body,
        meta_description=seo.meta_description,
        primary_keywords=seo.primary_keywords,
        fact_check_score=fact_check.overall_accuracy_score,
        readability_score=seo.readability_score,
        editorial_notes=editorial_notes,
    )

### 8. Full pipeline workflow

The complete workflow combines both patterns:
1. Sequential: `researcher_agent → writer_agent`
2. Parallel: `fact_checker_agent ‖ seo_optimizer_agent`
3. Sequential: `synthesizer_agent` (waits for both parallel tasks)

In the Flyte UI DAG view, the parallel tasks appear as a fork-join: two nodes pointing to the same downstream synthesizer node. Flyte schedules them concurrently automatically.

In [ ]:
@flyte.workflow
def full_content_pipeline(topic: str) -> FinalArticle:
    """
    Complete multi-agent content pipeline.

    DAG:
      researcher_agent(topic)
        └──► writer_agent(topic, research)
               ├──► fact_checker_agent(article)     ─┐
               │                                      ├──► synthesizer_agent → FinalArticle
               └──► seo_optimizer_agent(article) ─────┘

    Flyte guarantees:
    - researcher completes before writer starts
    - writer completes before fact_checker and seo_optimizer start
    - both fact_checker AND seo_optimizer complete before synthesizer starts
    - each failed task retries independently (no redundant re-runs)
    """
    # Phase 1: Sequential research → writing
    research = researcher_agent(topic=topic)
    article = writer_agent(topic=topic, research=research)

    # Phase 2: Parallel specialist review
    # Flyte sees that both tasks have the same input (article) and no
    # dependency on each other, so it schedules them concurrently.
    fact_check = fact_checker_agent(article=article)
    seo = seo_optimizer_agent(article=article)

    # Phase 3: Synthesis (waits for both parallel tasks)
    return synthesizer_agent(article=article, fact_check=fact_check, seo=seo)

### 9. Run locally

In [ ]:
run = flyte.with_runcontext(mode="local").run(
    full_content_pipeline,
    topic="How multi-agent AI systems are transforming enterprise software development",
)
run.wait()
final = run.outputs()[0]

print(f"Title: {final.title}")
print(f"Meta: {final.meta_description}")
print(f"Keywords: {', '.join(final.primary_keywords)}")
print(f"Fact-check score: {final.fact_check_score}/100")
print(f"Readability score: {final.readability_score}/100")
print(f"\nEditorial notes:")
for note in final.editorial_notes:
    print(f"  - {note}")
print("\n" + "=" * 60)
print(final.body)

### Running remotely

On a Flyte cluster, the full DAG is visible in the UI: researcher → writer → [fact-checker ‖ SEO optimizer] → synthesizer. Each node shows its typed inputs and outputs. The synthesizer shows a live report tab with the final article.

1. Create the secret on the cluster:

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

2. Switch to remote execution:

In [ ]:
if __name__ == "__main__":
    flyte.init_from_config()
    run = flyte.run(
        full_content_pipeline,
        topic="How multi-agent AI systems are transforming enterprise software development",
    )
    run.wait()
    final = run.outputs()[0]
    print(final.title)
    print(final.body)

## Scaling the pattern

### Adding more parallel specialists

The parallel fork-join pattern scales linearly. Adding a legal compliance agent or a tone-of-voice agent requires only:
1. A new typed output dataclass
2. A new `@agent_env.task` function
3. Adding it to the workflow and synthesizer signature

Flyte schedules all parallel tasks concurrently — no thread management or `asyncio.gather` boilerplate needed at the workflow level.

In [ ]:
# Example: adding a legal compliance agent to the parallel review phase

@dataclass
class ComplianceResult:
    issues: list[str]
    risk_level: str   # "low" | "medium" | "high"


@agent_env.task(retries=3, timeout=timedelta(minutes=10), cache=flyte.Cache(behavior="disable"))
async def compliance_agent(article: ArticleOutput) -> ComplianceResult:
    """Legal compliance reviewer — runs in parallel with fact-checker and SEO optimizer."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="You are a legal compliance reviewer. Identify any claims that could be legally problematic. Return JSON: {\"issues\": [...], \"risk_level\": \"low|medium|high\"}",
        messages=[{"role": "user", "content": article.body}],
    )
    import json
    data = json.loads(response.content[0].text.strip())
    return ComplianceResult(issues=data["issues"], risk_level=data["risk_level"])


# In the workflow, add compliance_agent alongside the other parallel tasks:
# compliance = compliance_agent(article=article)
# return synthesizer_agent(article=article, fact_check=fact_check, seo=seo, compliance=compliance)

### Supervisor pattern

For dynamic task delegation — where a coordinator decides *at runtime* which agents to call — implement a supervisor as a single Flyte task that dispatches to sub-tasks using `asyncio.gather`. The coordinator reads the task requirements and routes to appropriate specialists.

In [ ]:
@agent_env.task(
    retries=3,
    timeout=timedelta(minutes=20),
    cache=flyte.Cache(behavior="disable"),
)
async def supervisor_agent(topic: str, required_reviews: list[str]) -> dict:
    """
    Supervisor pattern: a coordinator that dynamically dispatches to specialists.

    Unlike the workflow-level parallelism above (where all agents always run),
    the supervisor can decide at runtime which specialists are needed based
    on the topic or policy requirements.

    Trade-off vs. workflow-level parallelism:
    - Pro: dynamic routing (e.g., only run compliance for regulated topics)
    - Con: all work happens in one task — if it fails, all sub-work retries
    Use workflow-level parallelism when the set of agents is fixed;
    use supervisor when routing logic needs to be dynamic.
    """
    # Phase 1: Research and write
    research = await _research(topic=topic)
    research_output = ResearchOutput(
        topic=topic,
        summary=research["summary"],
        key_findings=research["key_findings"],
        notable_examples=research["notable_examples"],
        recommended_angle=research["recommended_angle"],
        sources_consulted=research["sources_consulted"],
    )
    article_data = await _write(topic=topic, research=research_output)
    article = ArticleOutput(
        topic=topic,
        title=article_data["title"],
        body=article_data["body"],
        word_count=article_data.get("word_count", len(article_data["body"].split())),
        target_audience=article_data["target_audience"],
        key_points=article_data["key_points"],
    )

    # Phase 2: Dynamically dispatch only requested reviews in parallel
    review_coros = []
    if "fact_check" in required_reviews:
        review_coros.append(_call_specialist(FACT_CHECKER_SYSTEM, article.body))
    if "seo" in required_reviews:
        review_coros.append(_call_specialist(SEO_SYSTEM, article.body))

    review_results = await asyncio.gather(*review_coros)

    return {
        "article": article.body,
        "reviews": dict(zip(required_reviews, review_results)),
    }